app.py

In [ ]:
from flask import Flask, render_template, request, jsonify
from services.speech_to_text import transcribe_audio
import threading
import os

app = Flask(__name__)

progress = 0
transcript_result = ""
sentiment_result = ""
is_processing = False


def process_audio(path):
    global progress, transcript_result, sentiment_result, is_processing

    try:
        progress = 10

        transcript = transcribe_audio(path, update_progress)

        transcript_result = transcript
        sentiment_result = "Positive"  # nanti bisa diganti AI

        progress = 100

    except Exception as e:
        transcript_result = f"Error: {str(e)}"
        progress = 0

    is_processing = False


@app.route("/", methods=["GET", "POST"])
def index():
    global is_processing

    if request.method == "POST":

        if is_processing:
            return "Masih memproses audio sebelumnya..."

        audio = request.files["audio"]

        if audio.filename == "":
            return "File kosong"

        os.makedirs("uploads", exist_ok=True)

        path = "uploads/audio.wav"
        audio.save(path)

        is_processing = True

        # jalankan di background
        thread = threading.Thread(target=process_audio, args=(path,))
        thread.start()

    return render_template("index.html",
        transcript=transcript_result,
        sentiment=sentiment_result
    )


@app.route("/progress")
def get_progress():
    return jsonify({
        "progress": progress,
        "processing": is_processing
    })


def update_progress(value):
    global progress
    progress = min(value, 100)


if __name__ == "__main__":
    app.run(debug=True)

speect to text

In [ ]:
from faster_whisper import WhisperModel
import os

# load model (lebih cepat & ringan)
model = WhisperModel(
    "small", 
    compute_type="int8"
)

def transcribe_audio(audio_path, progress_callback):

    # validasi file
    if not os.path.exists(audio_path):
        return "File tidak ditemukan"

    if os.path.getsize(audio_path) == 0:
        return "Audio kosong"

    try:
        progress_callback(10)

        # transcribe + VAD (buang bagian hening)
        segments, info = model.transcribe(
            audio_path,
            language="id",
            beam_size=3,
            vad_filter=True
        )

        # ubah generator ke list supaya bisa hitung progress
        segments = list(segments)
        total_segments = len(segments)

        if total_segments == 0:
            return "Audio tidak terbaca / terlalu hening"

        text = ""

        for i, segment in enumerate(segments):
            text += segment.text.strip() + " "

            # progress real-time
            percent = 10 + int((i + 1) / total_segments * 80)
            progress_callback(percent)

        progress_callback(95)

        return text.strip()

    except Exception as e:
        return f"Error saat transkripsi: {str(e)}"